# Baseline Evaluation (fast track)

Speed-optimized variant of `kaggle_baselines.ipynb` for a bounded time budget.
Same baselines, same metrics; two changes make it finish in minutes instead of
many hours:

- **Read once, not per baseline.** The streaming notebook re-reads the whole test
  split from HDF5 once *per baseline* (10x), and each per-batch read reopens the
  `.h5` and runs a SQLite lookup (~0.5 s each on the Kaggle mount, so ~2 h per
  baseline). Here we read a random **capped** subset ONCE into RAM (`N_TEST_BATCHES`)
  and every baseline reuses it with zero further disk I/O.
- **Fit on a small sample.** The learned baselines are fit on only `N_TRAIN_BATCHES`
  (~25) random train sub-batches, not the full train split.

No large local copy is made (reads come straight from the mount), so it stays within
a small disk budget. GPU is used when present (`DEVICE=cuda`); one T4 is plenty.

**Caveat:** metrics are computed on a *capped, random* test subset, so they are an
estimate, not the full-split numbers. Raise `N_TEST_BATCHES` for tighter estimates
at the cost of a longer one-time read. `us/atom` is wall-clock (host+device),
comparable only within this run.


In [ ]:
import os, subprocess, sys

NOTEBOOK_NAME = "your-kaggle-notebook"  # set to your Kaggle notebook slug
assert NOTEBOOK_NAME != "your-kaggle-notebook", "Set NOTEBOOK_NAME first."

# Read straight from the Kaggle dataset mount. No local copy: the .h5 is large and
# disk is limited, and the fast notebook only ever reads a small capped subset ONCE
# (see the dataloader cell), so the slow per-read mount cost is paid once total.
HDF5_PATH = "/kaggle/input/datasets/noso0s0n/iql50/I(q)L50.h5"
DB_NAME = "/kaggle/input/datasets/noso0s0n/iql50/iq_train_set-ENCODING.sqlite3"
REPO = f"/kaggle/working/{NOTEBOOK_NAME}"

# Fast-track knobs.
N_BUCKETS = 57  # spread the sampled batches across all 57 size buckets
N_TRAIN_BATCHES = (
    25  # random train sub-batches used only to fit the learned baselines
)
N_TEST_BATCHES = (
    300  # random test sub-batches: the shared eval set for every baseline
)

# install deps
%pip install -q xraydb beartype jaxtyping hdf5plugin h5py "scikit-learn>=1.3"
!curl -fsSL https://rclone.org/install.sh | sudo bash

# plots render text through real LaTeX (xelatex) with JuliaMono; slow apt step (~2-3 min)
!sudo apt-get update -q && sudo apt-get install -y -q texlive-xetex texlive-latex-recommended texlive-fonts-recommended
!mkdir -p ~/.fonts && curl -fsSL https://github.com/cormullion/juliamono/releases/latest/download/JuliaMono-ttf.tar.gz \
    | tar -xz -C ~/.fonts && fc-cache -f ~/.fonts

# clone repo
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "https://github.com/noshou/APS360.git", REPO],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)

sys.path.insert(0, REPO)

In [ ]:
# ── rclone / Google Drive checkpointing ── run once per session ─────────────
# Reuses the same RCLONE_CONF Kaggle secret as kaggle_train.ipynb (it's just
# remote storage credentials, not training-specific). Checkpoints under a
# separate baselines_ckpts_fast/ prefix so this never collides with training
# checkpoints on the same Drive.
import base64, json
from kaggle_secrets import UserSecretsClient
from Baselines.metrics import EvalResult

conf_path = "/kaggle/working/rclone.conf"
with open(conf_path, "w") as f:
    f.write(
        base64.b64decode(
            UserSecretsClient().get_secret("RCLONE_CONF")
        ).decode()
    )
os.environ["RCLONE_CONFIG"] = conf_path

remotes = (
    subprocess.run(["rclone", "listremotes"], capture_output=True, text=True)
    .stdout.strip()
    .split("\n")
)
remote = remotes[0] if remotes and remotes[0] else ""
assert remote.endswith(":"), (
    f"No rclone remote found (rclone listremotes -> {remotes!r}). Check the RCLONE_CONF secret."
)
REMOTE_NAME = remote + f"{NOTEBOOK_NAME}/baselines_ckpts_fast/"

out = subprocess.run(
    ["rclone", "mkdir", REMOTE_NAME], capture_output=True, text=True
)
print(
    "remote drive ──>",
    REMOTE_NAME,
    "(ok)" if out.returncode == 0 else f"ERROR: {out.stderr.strip()}",
)

_CKPT_LOCAL = "/kaggle/working/baselines_results_fast.json"

_CKPT_REMOTE_NAME = os.path.basename(
    _CKPT_LOCAL
)  # "baselines_results_fast.json" -- must match on
# both ends, or save/load silently target
# different files and nothing ever resumes


def load_checkpoint() -> dict:
    """Pull the checkpoint from Drive if it exists and return completed baselines.

    Returns name -> EvalResult (the full result, including per-q arrays needed
    for plotting -- not just the summary numbers), so a resumed baseline still
    has plot data and R²(raw) available this session. Returns an empty dict on
    a fresh run (nothing to resume) or if the Drive pull failed; the printed
    message distinguishes the two. Baselines whose name is already a key here
    get skipped by the evaluation loops below.
    """
    out = subprocess.run(
        [
            "rclone",
            "copy",
            f"{REMOTE_NAME}{_CKPT_REMOTE_NAME}",
            "/kaggle/working/",
        ],
        capture_output=True,
        text=True,
    )
    if not os.path.exists(_CKPT_LOCAL):
        if out.returncode != 0:
            print(
                f"No existing checkpoint on Drive (or pull failed): {out.stderr.strip()}"
            )
        return {}
    with open(_CKPT_LOCAL) as f:
        raw = json.load(f)
    # Old checkpoint schema (pre EvalResult.to_json) stored each entry as a bare
    # [msle, r2_log1p, us_per_atom] list, not a dict -- from_json can't parse that
    # (and shouldn't try to: it's missing r2_raw and every per-q array). Drop those
    # entries instead of crashing, so the baseline just re-runs fresh this session.
    data, stale_schema = {}, []
    for name, v in raw.items():
        if isinstance(v, dict):
            data[name] = EvalResult.from_json(v)
        else:
            stale_schema.append(name)
    if stale_schema:
        print(
            f"Checkpoint has {len(stale_schema)} entry(ies) in the old pre-EvalResult "
            f"schema, dropping so they re-run: {stale_schema}"
        )
    print(
        f"Resumed {len(data)} completed baseline(s) from checkpoint: {list(data.keys())}"
    )
    return data


def save_checkpoint(results: dict) -> None:
    """Write the checkpoint locally and push it to Drive. Call after every baseline.

    Verifies the push with `rclone lsf` rather than trusting `copy`'s exit code --
    rclone can return 0 while nothing actually lands on Drive (stale/expired
    OAuth token, wrong root_folder_id, Shared-Drive permissions), which is the
    silent-failure mode that makes a checkpoint look saved when it never
    reached Drive at all.
    """
    with open(_CKPT_LOCAL, "w") as f:
        json.dump(
            {name: r.to_json() for name, r in results.items()}, f, indent=2
        )
    out = subprocess.run(
        ["rclone", "copy", _CKPT_LOCAL, REMOTE_NAME],
        capture_output=True,
        text=True,
    )
    if out.returncode != 0:
        print(
            f"WARNING: rclone push of checkpoint failed: {out.stderr.strip()}"
        )
        return
    verify = subprocess.run(
        ["rclone", "lsf", f"{REMOTE_NAME}{_CKPT_REMOTE_NAME}"],
        capture_output=True,
        text=True,
    )
    if verify.returncode != 0 or not verify.stdout.strip():
        print(
            f"WARNING: rclone copy exited 0 but the checkpoint isn't visible on Drive "
            f"afterward ({verify.stderr.strip() or 'empty listing'}) -- this session's "
            f"checkpoint is NOT actually saved to Drive; check RCLONE_CONF / Drive permissions"
        )
    else:
        print(f"checkpoint pushed to Drive ({len(results)} baseline(s) saved)")

In [ ]:
import h5py, hdf5plugin
from Preprocess.encode import Encoding
from ScatterNet.utils.config import DEFAULT_BUCKETS

print(
    "Loading encoding DB (iq_train_set-ENCODING.sqlite3, mounted from Kaggle dataset)..."
)
enc = Encoding(DB_NAME, HDF5_PATH)
print(f"  {enc.count():,} molecules  |  max atoms: {enc._max}")

with h5py.File(HDF5_PATH, "r") as f:
    q_grid = f["q_grid"][()]
    energy = float(f.attrs.get("energy", 10000.0))

import torch

q_grid = torch.from_numpy(q_grid).float()
print(f"  q_grid: {len(q_grid)} points  |  energy: {energy} eV")

In [ ]:
import random, time
from ScatterNet.batching import Batcher, Batch
from torch.utils.data import DataLoader, Subset

BUCKET_SAMPLE_SEED = (
    3092983  # deterministic bucket subset (keep == other notebooks)
)


def _first(x):
    return x[0]


def materialize(dataset, n_max, seed, name):
    """Read a random subset of <= n_max sub-batches from HDF5 ONCE into a RAM list.

    This is the whole point of the fast notebook. The streaming notebook re-reads the
    split from disk once PER baseline (10x), and every per-batch read reopens the .h5
    and runs a SQLite lookup (~0.5 s each on the Kaggle mount -> ~2 h per baseline).
    Materializing a capped subset once collapses that to a single read; every baseline
    then iterates this list with zero further disk I/O. Batches stay on CPU here;
    evaluate() moves each to the GPU per pass (cheap for a capped set).
    """
    n = len(dataset)
    idx = sorted(random.Random(seed).sample(range(n), min(n_max, n)))
    loader = DataLoader(
        Subset(dataset, idx), batch_size=1, collate_fn=_first, num_workers=0
    )
    out, t0 = [], time.time()
    for i, b in enumerate(loader):
        out.append(b)
        if (i + 1) % 10 == 0 or (i + 1) == len(idx):
            print(
                f"\r  materializing {name}: {i + 1}/{len(idx)}  ({time.time() - t0:.0f}s)",
                end="",
                flush=True,
            )
    print()
    return out


eval_buckets = sorted(
    random.Random(BUCKET_SAMPLE_SEED).sample(
        DEFAULT_BUCKETS, min(N_BUCKETS, len(DEFAULT_BUCKETS))
    )
)

batcher = Batcher(
    hdf5_db=HDF5_PATH,
    enc=enc,
    batches=eval_buckets,
    seed=42,
    atom_size_ceil=6046,
)
train_set, _, test_set = batcher.get_sets()

# one read pass total: train capped for fitting, test capped as the shared eval set
train_loader = materialize(train_set, N_TRAIN_BATCHES, 101, "train set")
test_loader = materialize(test_set, N_TEST_BATCHES, 202, "test set")
print(
    f"Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}"
)

In [ ]:
import torch
from Baselines.metrics import evaluate as _evaluate

# Run every baseline on the GPU when one is present (Kaggle 2x T4 -> cuda:0). The
# pairwise-sinc physics baselines (SAXS propoEst/stratEst especially) are O(atoms^2)
# and were hours each on CPU; on a single T4 they are minutes. evaluate() moves each
# batch to DEVICE and every baseline is device-following, so this is all that is
# needed. One GPU is more than enough here, so we use cuda:0 and leave the second
# T4 idle rather than add multi-GPU sharding to a run that is already minutes long.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Baselines device: {DEVICE}")


def evaluate(baseline, loader, name):
    """Evaluate a baseline, print its metrics, and return the full EvalResult.

    See Baselines/metrics.py for the full metric definitions (MSLE, R²
    raw/log1p, per-q breakdowns, etc). The full result (including the per-q
    arrays needed for plotting) is what gets checkpointed now, so a baseline
    resumed from checkpoint in a later session still plots and reports
    R²(raw) correctly.
    """
    result = _evaluate(baseline, loader, q_grid, name, device=DEVICE)
    print(
        f"{name:<30s}  MSLE={result.msle:.4f}  R²(raw)={result.r2_raw:.4f}  "
        f"R²(log1p)={result.r2_log1p:.4f}  {result.us_per_atom:.2f} μs/atom"
    )
    return result

In [ ]:
sys.path.insert(0, f"{REPO}/Baselines/physics-benchmarks")
sys.path.insert(0, f"{REPO}/Baselines/learned-benchmarks")

from rg import RgBaseline, GuinierPorodBaseline
from atom_count import AtomCountBaseline
from binned_debye import BinnedDebyeBaseline
from propo_est import PropoEstBaseline
from strat_est import StratEstBaseline

print("=== Physics Baselines ===")
results = load_checkpoint()

# train_loader / test_loader are the materialized (RAM) lists from the dataloader
# cell above -- no per-baseline re-read. factories, not instances: fit is deferred
# until we know a baseline isn't already in `results`.
physics_baselines = [
    ("Guinier (Rg)", lambda: RgBaseline(q_grid, energy)),
    ("Guinier-Porod", lambda: GuinierPorodBaseline(q_grid, energy)),
    ("Atom Count", lambda: AtomCountBaseline().fit(train_loader)),
    ("Binned Debye", lambda: BinnedDebyeBaseline(q_grid, energy)),
    ("SAXS propoEst", lambda: PropoEstBaseline(q_grid, energy, e=0.380)),
    ("SAXS stratEst", lambda: StratEstBaseline(q_grid, energy, a=0.6)),
]

for name, make_baseline in physics_baselines:
    if name in results:
        print(f"{name:<30s}  (skipped, resumed from checkpoint)")
        continue
    results[name] = evaluate(make_baseline(), test_loader, name)
    save_checkpoint(results)

In [ ]:
# self-contained: put the learned-benchmarks dir on sys.path here too, so this
# cell runs even when the physics cell above hasn't this session (fresh runtime,
# or run-from-here). Idempotent: a duplicate leading path entry is harmless.
sys.path.insert(0, f"{REPO}/Baselines/learned-benchmarks")

import torch
from mlp_2 import (
    Mlp2Baseline,
)  # Baselines/learned-benchmarks, added to sys.path above

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from linsvm import Linsvm
from nearest_neighbour import NNBaseline

print("=== Learned Baselines ===")

learned_baselines = [
    ("MLP", lambda: Mlp2Baseline().fit(train_loader)),
    ("Linear SVM", lambda: Linsvm().fit(train_loader)),
    (
        "Nearest Neighbour",
        lambda: NNBaseline(q_grid, energy).fit(train_loader),
    ),
]

for name, make_baseline in learned_baselines:
    if name in results:
        print(f"{name:<30s}  (skipped, resumed from checkpoint)")
        continue
    results[name] = evaluate(make_baseline(), test_loader, name)
    save_checkpoint(results)

In [ ]:
_known_names = {n for n, _ in physics_baselines} | {
    n for n, _ in learned_baselines
}
_stale = sorted(n for n in results if n not in _known_names)
if _stale:
    print(
        f"Dropping stale checkpoint entries no longer in the baseline list: {_stale}"
    )
    for n in _stale:
        del results[n]
    save_checkpoint(results)

print("\n=== Summary ===")
print(
    f"{'Baseline':<30s}  {'MSLE':>8s}  {'R²(raw)':>10s}  {'R²(log1p)':>12s}  {'μs/atom':>10s}"
)
print("-" * 78)
for name, r in sorted(results.items(), key=lambda x: x[1].msle):
    print(
        f"{name:<30s}  {r.msle:>8.4f}  {r.r2_raw:>10.4f}  {r.r2_log1p:>12.4f}  {r.us_per_atom:>10.2f}"
    )

In [ ]:
from Baselines.metrics import run_all_plots

_PLOTS_LOCAL = "/kaggle/working/baseline_plots"
_PLOTS_REMOTE = (
    remote + f"{NOTEBOOK_NAME}/baselines_ckpts_fast/baseline_plots/"
)

written = run_all_plots(list(results.values()), q_grid, _PLOTS_LOCAL)
print("wrote:", *written, sep="\n  ")

# /kaggle/working doesn't survive past the session -- push the plots to Drive
# the same way save_checkpoint() does for the results JSON (cell 2), or they're
# gone the moment this session ends even though the run itself succeeded.
out = subprocess.run(
    ["rclone", "copy", _PLOTS_LOCAL, _PLOTS_REMOTE],
    capture_output=True,
    text=True,
)
if out.returncode != 0:
    print(f"WARNING: rclone push of plots failed: {out.stderr.strip()}")
else:
    verify = subprocess.run(
        ["rclone", "lsf", _PLOTS_REMOTE], capture_output=True, text=True
    )
    n_remote = (
        len(verify.stdout.strip().splitlines()) if verify.stdout.strip() else 0
    )
    if verify.returncode != 0 or n_remote < len(written):
        print(
            f"WARNING: rclone copy exited 0 but only {n_remote}/{len(written)} plot(s) are "
            f"visible on Drive afterward ({verify.stderr.strip() or 'partial listing'})"
        )
    else:
        print(f"plots pushed to Drive ({n_remote} file(s)): {_PLOTS_REMOTE}")

In [ ]:
# per-q R², per-q percent error, Kratky overlay, residual histogram, and
# error-vs-atom-count are all written by run_all_plots() in the cell above --
# see Baselines/metrics.py for what each one shows and why.